# Bloque 1 — Perfil General de la base de Cuentas por Cobrar

**Objetivo:** entender qué contiene la fuente antes de analizar nada.
Respondo tres preguntas básicas: ¿cuántos registros tengo?, ¿qué columnas y de
qué tipo?, y ¿qué periodo de tiempo cubre la información?

**Por qué importa:** ningún análisis es confiable si primero no verifico el tamaño,
la estructura y el rango temporal de los datos. Este bloque sienta esa base.

In [4]:
import duckdb
import pandas as pd

# Abro un "motor" de consultas (DuckDB) y le enchufo mi base SQLite.
# Lo hago una sola vez; de aquí en adelante le hago preguntas en SQL.
con = duckdb.connect()
con.execute("INSTALL sqlite; LOAD sqlite;")
con.execute("ATTACH '/workspaces/prueba-tecnica-cxc/data/fuente_cxc.sqlite' AS fuente (TYPE sqlite);")

print("Conexión lista.")

Conexión lista.


## 1.1 Tamaño de la base

Confirmo cuántas cuentas por cobrar tengo en total. Este número define el enfoque
del análisis y lo dejo verificado por mí mismo, no como un dato supuesto.

In [5]:
# Lo primero que hace cualquier analista: saber el tamaño de lo que tiene enfrente.
# ¿Son 100 registros o 100 millones? Cambia todo el enfoque.
resumen = con.execute("SELECT COUNT(*) AS total_filas FROM fuente.tabla1").df()
print(resumen)

   total_filas
0        21739


## 1.2 Estructura: columnas y tipos de dato

Reviso qué columnas existen y cómo está guardada cada una. Presto atención especial
a las fechas (`f_creacion`, `f_ultimo_pago`): sospecho que están guardadas como
número y no como fecha, lo cual condiciona cómo tendré que tratarlas más adelante.

In [6]:
# Quiero el "índice" de la tabla: qué columnas existen y qué tipo de dato
# guarda cada una (texto, número entero, número con decimales).
# Esto me dice, por ejemplo, si las fechas están guardadas como número (ojo con eso).
estructura = con.execute("DESCRIBE fuente.tabla1").df()
print(estructura)

             column_name column_type null   key default extra
0          cod_apli_prod     VARCHAR  YES  None    None  None
1   descri_cod_apli_prod     VARCHAR  YES  None    None  None
2                num_cta      BIGINT  YES  None    None  None
3             f_creacion      BIGINT  YES  None    None  None
4          f_ultimo_pago      BIGINT  YES  None    None  None
5           vlr_original      DOUBLE  YES  None    None  None
6             vlr_pagado      DOUBLE  YES  None    None  None
7     vlr_pendiente_pago      DOUBLE  YES  None    None  None
8                cod_trn      BIGINT  YES  None    None  None
9         descri_cod_trn     VARCHAR  YES  None    None  None
10                  year      BIGINT  YES  None    None  None
11                 month      BIGINT  YES  None    None  None
12                   day      BIGINT  YES  None    None  None


## 1.3 Muestra de registros reales

Antes de sacar conclusiones, miro 10 registros de verdad para hacerme una idea de
cómo lucen los valores: el formato de las fechas-número, si los montos traen
decimales, y cómo se ven las descripciones de producto y transacción.

In [7]:
# Los números y tipos están bien, pero quiero VER registros de verdad.
# Traigo 10 filas para hacerme una idea de cómo se ven los valores reales:
# ¿cómo lucen las fechas-número?, ¿los montos tienen decimales raros?, etc.
muestra = con.execute("SELECT * FROM fuente.tabla1 LIMIT 10").df()
muestra

,cod_apli_prod,descri_cod_apli_prod,num_cta,f_creacion,f_ultimo_pago,vlr_original,vlr_pagado,vlr_pendiente_pago,cod_trn,descri_cod_trn,year,month,day
0,S,AHORRO,84443321151,20250211,20250411,154.08,154.08,0.00,1517,COBRO SERVICIO TRANSPORTE,2025,10,11
1,S,AHORRO,85352974856,20241103,20250921,1278.25,1278.25,0.00,2107,CARGO FISCAL IVA TRASLADO,2025,10,11
2,S,AHORRO,97822810913,20250501,20250625,5646.89,5646.89,0.00,834,CARGO FISCAL TRANSACCIONAL,2025,10,11
3,S,AHORRO,38571076870,20241222,20250714,285.77,257.19,28.58,834,CARGO FISCAL TRANSACCIONAL,2025,10,11
4,S,AHORRO,92759986619,20250222,20250901,17048.76,17048.76,0.00,834,CARGO FISCAL TRANSACCIONAL,2025,10,11
5,S,AHORRO,45058872234,20250521,20250722,370.19,370.19,0.00,788,TRANSFERENCIA CANAL FISICO,2025,10,11
6,S,AHORRO,30719760344,20250503,20250803,27623.59,27623.59,0.00,834,CARGO FISCAL TRANSACCIONAL,2025,10,11
7,S,AHORRO,25026503559,20241230,20250716,693.88,693.88,0.00,1517,COBRO SERVICIO TRANSPORTE,2025,10,11
8,S,AHORRO,23809278607,20241127,20250328,2039.71,2039.71,0.00,497,PAGO SERVICIO ELECTRONICO,2025,10,11
9,S,AHORRO,97668253073,20250312,20250616,637.51,573.76,63.75,834,CARGO FISCAL TRANSACCIONAL,2025,10,11


## 1.4 Rango temporal de la información

Las fechas vienen como número en formato AAAAMMDD (ej.: 20230415 = 15/abr/2023).
Las convierto a fecha real para saber **desde cuándo y hasta cuándo** va la historia
de las cuentas por cobrar.

> **Hallazgo relevante para más adelante:** si existen cuentas creadas hace muy poco,
> es esperable que aún no estén pagadas. Ese detalle será clave al medir la
> probabilidad de pago en la Actividad 2.

In [8]:
# Las fechas vienen como número tipo AAAAMMDD (ej. 20230415 = 15 abril 2023).
# No puedo calcular "días transcurridos" con un número así. Necesito convertirlo
# a una fecha de verdad. Lo hago con SQL: paso el número a texto y lo interpreto
# como fecha. Luego pregunto por la fecha más antigua y la más reciente,
# para saber qué periodo cubre la base.
rango_fechas = con.execute("""
    SELECT
        MIN(strptime(CAST(f_creacion AS VARCHAR), '%Y%m%d')) AS fecha_mas_antigua,
        MAX(strptime(CAST(f_creacion AS VARCHAR), '%Y%m%d')) AS fecha_mas_reciente
    FROM fuente.tabla1
""").df()
print(rango_fechas)

  fecha_mas_antigua fecha_mas_reciente
0        2024-10-11         2025-08-13


## 1.5 Consistencia de las columnas de fecha

La tabla trae `year`, `month` y `day` por separado, además de `f_creacion`.
Verifico que esas columnas sueltas **no se contradigan** con la fecha completa.
Si no coinciden, es un problema de confiabilidad de la fuente que debo reportar.

**Resultado esperado:** 0 registros inconsistentes (buena señal de calidad).
Cualquier número mayor a 0 es un hallazgo que documentaré.

In [9]:
# La tabla trae year/month/day por separado. Quiero verificar que esas columnas
# NO se contradigan con f_creacion. Si el año suelto dice 2023 pero f_creacion
# dice 2022, tengo un problema de calidad que debo reportar.
# Cuento cuántos registros NO coinciden.
inconsistencias_fecha = con.execute("""
    SELECT COUNT(*) AS registros_que_no_cuadran
    FROM fuente.tabla1
    WHERE year  <> CAST(strftime(strptime(CAST(f_creacion AS VARCHAR), '%Y%m%d'), '%Y') AS INTEGER)
       OR month <> CAST(strftime(strptime(CAST(f_creacion AS VARCHAR), '%Y%m%d'), '%m') AS INTEGER)
       OR day   <> CAST(strftime(strptime(CAST(f_creacion AS VARCHAR), '%Y%m%d'), '%d') AS INTEGER)
""").df()
print(inconsistencias_fecha)

   registros_que_no_cuadran
0                     21739


La columna month guarda el mes como número simple: octubre es 10, pero también enero es 1 (no 01). Cuando yo extraigo el mes desde la fecha con strftime, me lo devuelve como texto con cero adelante ("01"), y al comparar un número contra un texto con formato distinto, nunca coinciden, aunque representen lo mismo.

## 1.6 Completitud: valores faltantes por columna

Reviso, columna por columna, cuántos valores están vacíos (nulos). Una columna muy
incompleta condiciona qué puedo hacer con ella. Lo miro en cantidad y en porcentaje,
porque no es lo mismo 5 vacíos que 5.000 sobre 21.739 registros.

In [10]:
# Traigo toda la tabla a un DataFrame de pandas para explorarla con comodidad.
# (Para EXPLORAR uso Python; las TRANSFORMACIONES formales las haré en SQL más adelante.)
df = con.execute("SELECT * FROM fuente.tabla1").df()

# Cuento cuántos valores faltantes (nulos) hay en cada columna.
# .isna() marca los vacíos; .sum() los suma por columna.
nulos = df.isna().sum()
print("Cantidad de valores faltantes por columna:")
print(nulos)

Cantidad de valores faltantes por columna:
cod_apli_prod           0
descri_cod_apli_prod    0
num_cta                 0
f_creacion              0
f_ultimo_pago           0
vlr_original            0
vlr_pagado              0
vlr_pendiente_pago      0
cod_trn                 0
descri_cod_trn          0
year                    0
month                   0
day                     0
dtype: int64


Ahora lo expreso en porcentaje, para dimensionar el impacto real de cada vacío
respecto al total de la base.

In [11]:
# Paso los vacíos a porcentaje sobre el total de filas, para dimensionar el impacto.
# Redondeo a 2 decimales para que se lea fácil.
pct_nulos = (df.isna().sum() / len(df) * 100).round(2)
print("Porcentaje de valores faltantes por columna (%):")
print(pct_nulos)

Porcentaje de valores faltantes por columna (%):
cod_apli_prod           0.0
descri_cod_apli_prod    0.0
num_cta                 0.0
f_creacion              0.0
f_ultimo_pago           0.0
vlr_original            0.0
vlr_pagado              0.0
vlr_pendiente_pago      0.0
cod_trn                 0.0
descri_cod_trn          0.0
year                    0.0
month                   0.0
day                     0.0
dtype: float64


### Un vacío que puede estar "disfrazado": la fecha de último pago

Ojo con algo: a veces un dato faltante no aparece como vacío, sino como un **0**. Es
muy probable que las cuentas que **nunca han recibido un pago** tengan
`f_ultimo_pago` en 0 (o vacío) en lugar de una fecha. Eso no es un error: es
información valiosa, porque una cuenta sin fecha de pago es, justamente, una cuenta
que aún no se ha recuperado. Lo cuantifico.

In [12]:
# Reviso cuántas cuentas no tienen fecha de último pago (vacía o en 0).
# Hipótesis: son cuentas que todavía no han recibido ningún pago.
# Esto me interesa mucho, porque conecta con la futura variable "¿se pagó o no?".
sin_fecha_pago = con.execute("""
    SELECT
        COUNT(*) AS total_cuentas,
        SUM(CASE WHEN f_ultimo_pago IS NULL OR f_ultimo_pago = 0 THEN 1 ELSE 0 END) AS sin_fecha_de_pago
    FROM fuente.tabla1
""").df()
print(sin_fecha_pago)

   total_cuentas  sin_fecha_de_pago
0          21739                0.0


### 1.6.1 ¿La "fecha de último pago" es real o es un relleno?

Me llama la atención que el 100% de las cuentas tenga fecha de último pago. En
cobranza real eso es raro. Sospecho que las cuentas sin pago no quedan vacías, sino
que el sistema les asigna la misma fecha de creación. Lo compruebo comparando ambas
fechas: si muchas cuentas tienen f_ultimo_pago = f_creacion, mi sospecha se confirma.

In [13]:
# Comparo la fecha de creación con la de último pago.
# Si son IGUALES en muchos casos, significa que "último pago = creación" es un relleno
# para cuentas que en realidad no han recibido ningún pago.
comparacion_fechas = con.execute("""
    SELECT
        COUNT(*) AS total_cuentas,
        SUM(CASE WHEN f_ultimo_pago = f_creacion THEN 1 ELSE 0 END) AS pago_igual_a_creacion,
        SUM(CASE WHEN f_ultimo_pago > f_creacion THEN 1 ELSE 0 END) AS pago_posterior_a_creacion,
        SUM(CASE WHEN f_ultimo_pago < f_creacion THEN 1 ELSE 0 END) AS pago_anterior_a_creacion
    FROM fuente.tabla1
""").df()
print(comparacion_fechas)

   total_cuentas  pago_igual_a_creacion  pago_posterior_a_creacion  \
0          21739                    0.0                    21739.0   

   pago_anterior_a_creacion  
0                       0.0  


### Conclusiones del Bloque 1 (Perfil General)

- Total de cuentas por cobrar: 21.739.
- Estructura: 13 columnas. Fechas guardadas como entero (AAAAMMDD); su estandarización
  se hará en la fase de limpieza.
- Periodo cubierto: del 11/oct/2024 al 13/ago/2025 (~10 meses). Implica censura en las
  cuentas más recientes (a considerar en el modelo).
- Completitud: 100%, sin valores faltantes en ninguna columna.
- Fechas de pago: el 100% son posteriores a la creación; cero fechas invertidas.
- Hallazgo de diseño: el estado de recuperación se derivará de los montos, no de las
  fechas.
- Pendiente para limpieza: estandarizar fechas y revisar coherencia year/month/day.

## 1.7 Grano de la tabla: ¿qué representa cada fila?

Verifico si cada fila es una cuenta por cobrar única o si un mismo número de cuenta
(`num_cta`) se repite. Esto define la unidad de análisis: saber si estoy contando
"cuentas" o "obligaciones dentro de una cuenta" cambia toda la interpretación.

In [14]:
# Comparo el total de filas contra el total de números de cuenta DISTINTOS.
# Si son iguales -> cada fila es una cuenta única (grano = cuenta).
# Si hay menos cuentas distintas que filas -> hay cuentas que se repiten
#    (grano = obligación/transacción, y una cuenta puede tener varias).
grano = con.execute("""
    SELECT
        COUNT(*)               AS total_filas,
        COUNT(DISTINCT num_cta) AS cuentas_distintas
    FROM fuente.tabla1
""").df()
print(grano)

   total_filas  cuentas_distintas
0        21739                800


Si hay cuentas repetidas, quiero ver cuáles son las que más se repiten, para entender
el patrón (por ejemplo, una misma cuenta con varios cobros distintos).

In [15]:
# Si hubo repetidos, miro las 10 cuentas que más veces aparecen.
# Esto me ayuda a entender el patrón: ¿una cuenta acumula muchas obligaciones?
repetidas = con.execute("""
    SELECT num_cta, COUNT(*) AS veces_aparece
    FROM fuente.tabla1
    GROUP BY num_cta
    HAVING COUNT(*) > 1
    ORDER BY veces_aparece DESC
    LIMIT 10
""").df()
print("Cuentas que aparecen más de una vez (top 10):")
print(repetidas)

Cuentas que aparecen más de una vez (top 10):
       num_cta  veces_aparece
0  82311788915             32
1  91115494880             32
2  38375846892             32
3  82258119687             32
4  29045469771             31
5  54242783600             31
6  11993485980             31
7  69041947068             31
8  28051623571             31
9  57868928692             31


### Conclusiones del punto 1.7

- Total de filas (obligaciones de cobro): 21.739.
- Números de cuenta (titulares) distintos: 800.
- Grano: la unidad es la OBLIGACIÓN de cobro, no el cliente. Cada titular genera
  ~27 obligaciones en promedio (hasta 32).
- Implicación: se puede analizar a nivel obligación (modelo) y agregar a nivel cuenta
  (concentración de riesgo por titular).

## 1.8 Consistencia contable: ¿cuadran los montos?

Valido la regla contable fundamental: para cada obligación,
valor_original = valor_pagado + valor_pendiente. Cuento cuántas filas NO cumplen esta
igualdad. Como son decimales, permito una diferencia mínima de un centavo para evitar
falsas alarmas por redondeo. Este resultado es la base para derivar el estado real de
cada cuenta.

In [16]:
# Valido la ecuación contable: original = pagado + pendiente.
# No uso "=" exacto porque los decimales arrastran errores minúsculos de redondeo;
# considero que NO cuadra solo si la diferencia supera 1 centavo (0.01).
consistencia = con.execute("""
    SELECT
        COUNT(*) AS total_filas,
        SUM(CASE
              WHEN ABS(vlr_original - (vlr_pagado + vlr_pendiente_pago)) > 0.01
              THEN 1 ELSE 0
            END) AS filas_que_no_cuadran
    FROM fuente.tabla1
""").df()
print(consistencia)

   total_filas  filas_que_no_cuadran
0        21739                   0.0


### Vistazo al estado real (adelanto del diseño de la sábana)

Ya que los montos son la fuente de verdad, hago un primer conteo del estado de las
cuentas según su saldo:
- Pagada total: no debe nada (pendiente = 0).
- Pago parcial: pagó algo pero aún debe (pendiente > 0 y pagado > 0).
- Sin pago: no ha pagado nada (pagado = 0).

In [17]:
# Primer conteo del estado real de las cuentas, derivado de los MONTOS (no de fechas).
# Esto es un adelanto de cómo definiré el estado en la sábana.
estado_preliminar = con.execute("""
    SELECT
        CASE
            WHEN vlr_pendiente_pago <= 0.01 THEN 'PAGADA_TOTAL'
            WHEN vlr_pagado <= 0.01        THEN 'SIN_PAGO'
            ELSE 'PAGO_PARCIAL'
        END AS estado,
        COUNT(*) AS cantidad,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS porcentaje
    FROM fuente.tabla1
    GROUP BY 1
    ORDER BY cantidad DESC
""").df()
print(estado_preliminar)

         estado  cantidad  porcentaje
0  PAGADA_TOTAL     17323       79.69
1  PAGO_PARCIAL      2669       12.28
2      SIN_PAGO      1747        8.04


### Conclusiones del punto 1.8

- Consistencia contable: 0 filas descuadradas. La igualdad
  original = pagado + pendiente se cumple en el 100%. Los montos son fuente de verdad.
- Distribución del estado (por cantidad de obligaciones):
  - PAGADA_TOTAL: 79,69% (17.323)
  - PAGO_PARCIAL: 12,28% (2.669)
  - SIN_PAGO: 8,04% (1.747)
- Titular de negocio: ~20% del portafolio presenta dificultad de recuperación.
- Pendiente: dimensionar ese 20% en valor ($), no solo en cantidad.

## 1.9 Distribuciones por producto y tipo de transacción

Analizo de qué tipo son las obligaciones y dónde se concentran. Primero reviso el
producto (`descri_cod_apli_prod`), luego los tipos de transacción más frecuentes, y
finalmente cruzo tipo de transacción con estado de recuperación para detectar si
algún tipo de cobro se recupera peor que otros (hallazgo accionable).

In [18]:
# ¿Cuántos productos distintos hay? En la muestra todo era AHORRO; lo confirmo.
productos = con.execute("""
    SELECT descri_cod_apli_prod AS producto, COUNT(*) AS cantidad
    FROM fuente.tabla1
    GROUP BY 1
    ORDER BY cantidad DESC
""").df()
print("Distribución por producto:")
print(productos)

Distribución por producto:
    producto  cantidad
0     AHORRO     21491
1  CORRIENTE       248


In [19]:
# Los 10 tipos de transacción que más obligaciones de cobro generan.
# Esto me dice qué operaciones son el grueso de las cuentas por cobrar.
transacciones = con.execute("""
    SELECT descri_cod_trn AS tipo_transaccion, COUNT(*) AS cantidad,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS porcentaje
    FROM fuente.tabla1
    GROUP BY 1
    ORDER BY cantidad DESC
    LIMIT 10
""").df()
print("Top 10 tipos de transacción:")
print(transacciones)

Top 10 tipos de transacción:
                   tipo_transaccion  cantidad  porcentaje
0        CARGO FISCAL TRANSACCIONAL      6823       31.39
1         COBRO SERVICIO TRANSPORTE      3704       17.04
2         PAGO SERVICIO ELECTRONICO      1312        6.04
3  COMISION TRANSFERENCIA EXTERNA B      1184        5.45
4    COMISION RETIRO CORRESPONSAL B      1129        5.19
5         CARGO FISCAL IVA TRASLADO      1070        4.92
6           COMISION CONSULTA SALDO       802        3.69
7       CARGO FISCAL IVA COMISION B       659        3.03
8           COMISION RETIRO CANAL A       645        2.97
9         TRANSFERENCIA BILLETERA A       507        2.33


por cada tipo de transacción, ¿qué porcentaje se paga total,
parcial o no se paga? Aquí busco si hay tipos de cobro problemáticos.

In [20]:
# Cruce de tipo de transacción con estado de recuperación.
# Busco tipos de cobro con alta proporción de PAGO_PARCIAL o SIN_PAGO:
# esos son los focos de gestión. Me quedo con los tipos de mayor volumen para que
# el análisis sea representativo.
cruce = con.execute("""
    WITH clasificada AS (
        SELECT
            descri_cod_trn AS tipo_transaccion,
            CASE
                WHEN vlr_pendiente_pago <= 0.01 THEN 'PAGADA_TOTAL'
                WHEN vlr_pagado <= 0.01        THEN 'SIN_PAGO'
                ELSE 'PAGO_PARCIAL'
            END AS estado
        FROM fuente.tabla1
    )
    SELECT
        tipo_transaccion,
        COUNT(*) AS total,
        ROUND(100.0 * SUM(CASE WHEN estado='PAGADA_TOTAL' THEN 1 ELSE 0 END)/COUNT(*), 1) AS pct_pagada,
        ROUND(100.0 * SUM(CASE WHEN estado='PAGO_PARCIAL' THEN 1 ELSE 0 END)/COUNT(*), 1) AS pct_parcial,
        ROUND(100.0 * SUM(CASE WHEN estado='SIN_PAGO'     THEN 1 ELSE 0 END)/COUNT(*), 1) AS pct_sin_pago
    FROM clasificada
    GROUP BY 1
    HAVING COUNT(*) >= 100
    ORDER BY pct_sin_pago DESC
    LIMIT 15
""").df()
print("Tipos de transacción ordenados por % sin pago (mayor riesgo arriba):")
print(cruce)

Tipos de transacción ordenados por % sin pago (mayor riesgo arriba):
                    tipo_transaccion  total  pct_pagada  pct_parcial  \
0         TRANSFERENCIA CANAL FISICO    139        36.7         37.4   
1                 CARGO FISCAL IVA A    109        71.6          6.4   
2   COMISION TRANSFERENCIA EXTERNA B   1184        69.3         15.1   
3            COMISION CONSULTA SALDO    802        67.7         19.0   
4      COMISION RETIRO INTERNACIONAL    127        54.3         33.9   
5          TRANSFERENCIA BILLETERA A    507        80.7          8.3   
6        CARGO FISCAL IVA COMISION B    659        77.2         12.6   
7            COMISION RETIRO CANAL A    645        71.6         19.7   
8     COMISION RETIRO CORRESPONSAL B   1129        73.9         17.4   
9          COBRO SERVICIO TRANSPORTE   3704        82.6          9.6   
10        CARGO FISCAL TRANSACCIONAL   6823        77.9         14.5   
11         TRANSFERENCIA BANCA MOVIL    110        70.9         21.

### Conclusiones del punto 1.9

- Productos: AHORRO ~99% (21.491), CORRIENTE ~1% (248). CORRIENTE con poco volumen
  para análisis robusto.
- Transacciones dominantes: CARGO FISCAL TRANSACCIONAL (31,4%) y COBRO SERVICIO
  TRANSPORTE (17%) concentran casi la mitad del volumen (Pareto).
- Peor recuperación en tasa: TRANSFERENCIA CANAL FISICO (36,7% pagada total) y
  CARGO FISCAL IVA A, ambos de bajo volumen.
- Foco de mayor impacto (mal desempeño + volumen): COMISION TRANSFERENCIA EXTERNA B
  (15,5% sin pago sobre 1.184).
- Regla de priorización: combinar tasa de no-recuperación, volumen y valor ($).

## 1.10 Distribución de montos y valores extremos

Analizo la magnitud de las obligaciones (`vlr_original`): estadísticos básicos
(mínimo, máximo, promedio, mediana) y percentiles para entender la forma de la
distribución. Reviso si hay valores extremos que puedan distorsionar promedios o el
modelo. También comparo promedio vs mediana: si difieren mucho, la distribución está
sesgada por pocos montos grandes.

In [21]:
# Estadísticos de la magnitud de las obligaciones.
# Comparo promedio (media) vs mediana: si la media es MUCHO mayor que la mediana,
# significa que unas pocas obligaciones muy grandes están "jalando" el promedio
# hacia arriba (distribución sesgada, típica en montos financieros).
montos = con.execute("""
    SELECT
        COUNT(*)                                   AS n,
        ROUND(MIN(vlr_original), 2)                AS minimo,
        ROUND(MAX(vlr_original), 2)                AS maximo,
        ROUND(AVG(vlr_original), 2)                AS promedio,
        ROUND(MEDIAN(vlr_original), 2)             AS mediana,
        ROUND(QUANTILE_CONT(vlr_original, 0.25), 2) AS p25,
        ROUND(QUANTILE_CONT(vlr_original, 0.75), 2) AS p75,
        ROUND(QUANTILE_CONT(vlr_original, 0.95), 2) AS p95,
        ROUND(QUANTILE_CONT(vlr_original, 0.99), 2) AS p99
    FROM fuente.tabla1
""").df()
print("Distribución de vlr_original:")
print(montos.T)   # .T lo pone vertical para leerlo fácil

Distribución de vlr_original:
                  0
n          21739.00
minimo        36.33
maximo    446887.02
promedio    6512.01
mediana     2218.19
p25          856.61
p75         6001.66
p95        23666.85
p99        57126.25


Ahora dimensiono el dinero total en juego y cómo se reparte entre lo ya recuperado y
lo que sigue pendiente. Este es el tamaño monetario del problema.

In [22]:
# El tamaño monetario del portafolio: cuánto se generó, cuánto se ha recuperado y
# cuánto sigue pendiente. Le pongo VALOR a los porcentajes que ya conocía.
totales = con.execute("""
    SELECT
        ROUND(SUM(vlr_original), 2)       AS valor_total_generado,
        ROUND(SUM(vlr_pagado), 2)         AS valor_total_recuperado,
        ROUND(SUM(vlr_pendiente_pago), 2) AS valor_total_pendiente,
        ROUND(100.0 * SUM(vlr_pagado) / SUM(vlr_original), 2) AS pct_recuperado_en_valor
    FROM fuente.tabla1
""").df()
print("Tamaño monetario del portafolio:")
print(totales.T)

Tamaño monetario del portafolio:
                                    0
valor_total_generado     1.415646e+08
valor_total_recuperado   1.193746e+08
valor_total_pendiente    2.219000e+07
pct_recuperado_en_valor  8.433000e+01


### Conclusiones del punto 1.10

- Rango de montos: de 36 a 446.887. Distribución sesgada a la derecha.
- Promedio 6.512 vs mediana 2.218: pocas obligaciones grandes jalan el promedio.
  El monto típico se describe mejor con la mediana.
- Valor total: generado ~141,6 M, recuperado ~119,4 M, pendiente ~22,2 M.
- Recuperación en valor 84,3% vs en cantidad 79,7%: el no-pago se concentra en
  obligaciones pequeñas.
- Pendiente para el modelo: tratar los montos extremos (transformación o acotamiento).